# Prompt caching with OpenAI GPT on Amazon Bedrock

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

Prompt caching reuses the processed form of a long prefix that repeats across requests,
such as instructions, tool definitions or a reference document. Cached input is billed
at a lower rate than fresh input, and the model does not process it again. The GPT-6 and
GPT-5.6 models take explicit cache breakpoints that you place. GPT-5.5 and GPT-5.4 cache
on their own when they choose to, and take no cache parameters. This notebook calls
`bedrock-runtime`; the same requests work on `bedrock-mantle`.

**Covered:** a cacheable prefix, explicit breakpoints, reading the usage as cost, and
automatic caching on GPT-5.5 and GPT-5.4.

## Setup

Install the pinned requirements first (`pip install -r ../requirements.txt`, Python 3.11
or later). The notebook calls Bedrock with the OpenAI SDK, and uses boto3 to list the
inference profiles, because `bedrock-runtime` has no `GET /models`.

Endpoints, authentication and URL paths are explained in
[`00-foundations/01`](../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb).

In [1]:
import boto3
from aws_bedrock_token_generator import provide_token
from openai import OpenAI

REGION = "us-east-1"
MODEL = "us.openai.gpt-6-sol"  # explicit breakpoints
AUTO = "us.openai.gpt-5.5"  # automatic caching only

# bedrock-runtime's OpenAI-compatible path, with a short-term Bedrock API key minted
# from your AWS credentials. Mint it when you build the client, because the key expires.
client = OpenAI(
    base_url=f"https://bedrock-runtime.{REGION}.amazonaws.com/openai/v1",
    api_key=provide_token(region=REGION),
    timeout=120,
)

# bedrock-runtime has no GET /models, so check the inference profiles instead.
profiles = {
    p["inferenceProfileId"]
    for page in boto3.client("bedrock", region_name=REGION).get_paginator("list_inference_profiles").paginate()
    for p in page["inferenceProfileSummaries"]
}
for model in (MODEL, AUTO):
    print(f"{model:20} {'available' if model in profiles else 'no such profile in ' + REGION}")

us.openai.gpt-6-sol  available
us.openai.gpt-5.5    available


## 1. A cacheable prefix

A prefix shorter than 1,024 tokens is not cached, and nothing in the response says so: a
468-token prefix read no cached tokens on any repeat in September 2026 runs. The cell
builds a handbook of about 1,500 tokens; the usage in the next cell gives the exact
count. The cache matches an exact prefix, so static content goes first and anything that
varies goes after it.

In [2]:
import secrets

SECTIONS = """1. Service tiers. Every inference request may name a service tier. Priority is for
customer-facing, latency-critical paths. Standard is the default for everyday work. Flex
is discounted and suits evaluations, batch-shaped work and side tasks that can wait.

2. Retry policy. Clients retry with exponential backoff and jitter. HTTP 429 means
throttling and 5xx means capacity shedding; both are transient and are retried. Any
other 4xx is a client defect and is not retried.

3. Data handling. Every request sets its data-retention posture explicitly. Workloads
with regulated data run with zero data retention, and keep no conversation state on the
server; the client replays the history instead.

4. Cost attribution. Every workload runs under its own project, tagged with application,
environment, owner and cost centre. Untagged usage is charged to a shared pool.
"""
# A random copy number makes the prefix new on every run, so the first call below writes
# the cache instead of reading what an earlier run left there.
HANDBOOK = f"Platform handbook, copy {secrets.token_hex(4)}.\n\n" + SECTIONS * 8

print(f"handbook: {len(HANDBOOK)} characters")

handbook: 6971 characters


## 2. Explicit breakpoints

Put `prompt_cache_breakpoint` on the content block where the reusable prefix ends, and
send `prompt_cache_options` with the request. The SDK has no typed argument for the
options, so they go in `extra_body`. With `mode: "explicit"` only your breakpoints
count, and a request without one caches nothing. `implicit` puts the breakpoint at the
end of the latest message, which makes the varying question part of the cached span: as
of October 2026, each of four calls over one prefix read 0 cached tokens and wrote the
whole prefix again once the question length varied, where the same four questions with a
breakpoint after the handbook wrote once and then read 1,494 tokens on each of the other
three. `implicit` is the default, so it is also what a request carrying no
`prompt_cache_options` gets: four such calls over one prefix, with no options sent and
the question varying, read 0 cached tokens and wrote the whole prefix each time, the
same as asking for `implicit` by name. So send `mode: "explicit"` as well as placing
the breakpoint. A breakpoint on its own is not enough: with the breakpoint but no
options, the prefix was read back and the automatic breakpoint on the latest message
wrote the varying tail as well, 16, 24 and 13 tokens on the three calls after the
first. A request is capped at four cache writes, so that automatic breakpoint also
spends one of them; see the Bedrock
[prompt-caching page](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-caching.html).

`ttl` inside `prompt_cache_options` accepts only `30m`. Longer retention is a separate
top-level parameter, `prompt_cache_retention`, which takes `in_memory` or `24h`. A
request can also carry a top-level `prompt_cache_key`, which selects the cache it
matches against: as of October 2026 two requests sharing a prefix but sending different
keys each wrote their own copy and read 0 cached tokens, so a key that varies per
request loses the cache and returns no error. The first call writes the prefix
(`cache_write_tokens`) and a repeat within the retention window reads it
(`cached_tokens`).

In [3]:
def ask(question: str) -> tuple[str, dict]:
    """One request whose reusable prefix is the handbook, with a breakpoint after it."""
    response = client.responses.create(
        model=MODEL,
        input=[{"role": "user", "content": [
            {"type": "input_text", "text": HANDBOOK, "prompt_cache_breakpoint": {"mode": "explicit"}},
            {"type": "input_text", "text": question},  # can vary without breaking the match
        ]}],
        store=False,
        max_output_tokens=500,
        extra_body={"prompt_cache_options": {"mode": "explicit", "ttl": "30m"}},
    )
    details = response.usage.input_tokens_details
    usage = {
        "input": response.usage.input_tokens,
        "cached": details.cached_tokens or 0,
        "written": getattr(details, "cache_write_tokens", 0) or 0,
    }
    return response.output_text.strip(), usage


first_answer, first = ask("Which tier suits evaluations? One line.")
second_answer, second = ask("Which HTTP errors must not be retried? One line.")
print("call 1:", first, "|", first_answer[:70])
print("call 2:", second, "|", second_answer[:70])
share = second["cached"] / second["input"] if second["input"] else 0
print(f"\ncall 2 read {second['cached']} of its {second['input']} input tokens from the cache ({share:.0%})")

call 1: {'input': 1508, 'cached': 0, 'written': 1494} | Flex suits evaluations.
call 2: {'input': 1512, 'cached': 1494, 'written': 0} | Any 4xx error other than HTTP 429 must not be retried.

call 2 read 1494 of its 1512 input tokens from the cache (99%)


## 3. Reading the usage as cost

`usage.input_tokens` counts every input token. Of those, `cached_tokens` were read from
the cache and `cache_write_tokens` were written to it, and each class has its own rate
on the [Amazon Bedrock pricing page](https://aws.amazon.com/bedrock/pricing/). Split the
input three ways and multiply each part by its rate; the split below is what you would
price.

In [4]:
def input_split(usage: dict) -> dict:
    """Input tokens by how they are billed: fresh, read from the cache, written to it."""
    fresh = usage["input"] - usage["cached"] - usage["written"]
    return {"fresh": fresh, "cache read": usage["cached"], "cache write": usage["written"]}


for label, usage in (("call 1", first), ("call 2", second)):
    print(label, input_split(usage))

call 1 {'fresh': 14, 'cache read': 0, 'cache write': 1494}
call 2 {'fresh': 18, 'cache read': 1494, 'cache write': 0}


## 4. Automatic caching on GPT-5.5 and GPT-5.4

GPT-5.5 and GPT-5.4 take no cache parameters: `prompt_cache_breakpoint` returns 400
`prompt_cache_breakpoint is not supported with the 'us.openai.gpt-5.5' model.`, and
`prompt_cache_options` returns 400 `prompt_cache_options is not supported on this
model`. They may cache a prefix of 1,024 tokens or more on their own, with no write fee.
It is best-effort: in September 2026 runs, which call first read the cache varied from
run to run, one run had no hit within four calls, and a repeat after a hit sometimes
missed. Read `cached_tokens` to see what happened.

In [5]:
hits = []
for n in range(1, 5):
    response = client.responses.create(
        model=AUTO,
        input=f"{HANDBOOK}\n\nQuestion {n}: which tier is the default? One word.",
        store=False,
        max_output_tokens=200,
    )
    hits.append(response.usage.input_tokens_details.cached_tokens or 0)
    print(f"call {n}: input={response.usage.input_tokens}  cached={hits[-1]}  answer={response.output_text.strip()!r}")

first_hit = next((n for n, cached in enumerate(hits, 1) if cached), None)
print(f"\nfirst cache hit on call {first_hit}" if first_hit else "\nno cache hit in four calls")

call 1: input=1509  cached=0  answer='Standard'


call 2: input=1509  cached=0  answer='Standard'


call 3: input=1509  cached=0  answer='Standard'


call 4: input=1509  cached=0  answer='Standard'

no cache hit in four calls


## Things that differ

What trips people up with these models, and what to do about it.

| Behaviour | What you see | What to do |
|---|---|---|
| Short prefixes | `cached_tokens` stays 0 below 1,024 tokens, with no error | Make the reusable prefix at least 1,024 tokens (section 1). |
| Cache parameters on GPT-5.5 and GPT-5.4 | 400 `prompt_cache_breakpoint is not supported with the 'us.openai.gpt-5.5' model.` or `prompt_cache_options is not supported on this model` | Send neither to those models. They cache on their own. |
| `prompt_cache_options.ttl` | 400 `Invalid value: '24h'. Supported values are: '30m'.` | Use `30m` here, and `prompt_cache_retention` when you want longer. |
| `prompt_cache_retention` | 400 `Invalid value: '1h'. Supported values are: 'in_memory' and '24h'.` | Use `in_memory` or `24h`, as a top-level parameter rather than inside `prompt_cache_options`. |
| `explicit` mode without a breakpoint | `cached_tokens` and `cache_write_tokens` are both 0, so nothing is cached and no write is charged | Add a breakpoint where the prefix ends (section 2). |
| `implicit` mode, which is the default, under a question that varies | 0 cached tokens and a full prefix write on each of four calls (October 2026), because the breakpoint lands at the end of the latest message. A request sending no `prompt_cache_options` behaves the same way | Place an explicit breakpoint after the static prefix and send `prompt_cache_options.mode: "explicit"`, which also stops the automatic breakpoint from spending one of the four cache writes a request is capped at (section 2). |
| `prompt_cache_key` | A request sending a different key, or no key, reads 0 cached tokens from a prefix another key wrote, with no error | Send the same key for the requests that should share a prefix. |
| Prefix order | With the question before the reusable text, every call wrote the prefix again and none read it (September 2026) | Put static content first and the question last. |
| Automatic caching | A hit on some repeats and not others | Use a model with explicit breakpoints when the saving matters. |
| Model support | Explicit breakpoints worked on GPT-6 Sol, Luna and Astra and GPT-5.6 Sol, Terra and Luna (September 2026) | Check the model table in [`01-responses-api-core`](01-responses-api-core.ipynb). |

## Next

- The Responses API and the model list:
  [`01-responses-api-core`](01-responses-api-core.ipynb)
- Tool loops, whose instructions and tool definitions are the prefix to cache:
  [`03-tools-and-structured-output`](03-tools-and-structured-output.ipynb)
- Claude's cache controls:
  [`02-anthropic-claude/02`](../02-anthropic-claude/02-thinking-tools-and-caching.ipynb)
- The open-weight gpt-oss models:
  [`14-openai-gpt-oss/01`](../14-openai-gpt-oss/01-gpt-oss-on-both-endpoints.ipynb)
- Choosing a model and API across families:
  [`99-cross-cutting/01`](../99-cross-cutting/01-choosing-a-model-and-api.ipynb)
- Quotas, retries and latency:
  [`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)
- Production hardening:
  [`99-cross-cutting/03`](../99-cross-cutting/03-production-hardening.ipynb)